# Episode-bootstrap test of the time arrow

Training and test episodes are resampled with replacement. Each replicate refits a full regional ridge operator to forward and episode-reversed, linearly detrended trajectories.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig
ROOT = Path('..').resolve()
RUN = ROOT / 'logs/scripts/regional_time_arrow_bootstrap/initial'
OUT = ROOT / 'figures/261027时间箭头自助检验'
OUT.mkdir(parents=True, exist_ok=True)
scores = pd.read_csv(RUN / 'bootstrap_scores.csv')
summary = pd.read_csv(RUN / 'bootstrap_summary.csv')
summary

,state,direction,median,q025,q975,positive_fraction
0,Awake,forward,0.074053,0.017078,0.105084,0.985
1,Awake,forward_minus_reversed,-0.024842,-0.054898,-0.010106,0.000
2,Awake,reversed,0.097869,0.061553,0.126622,1.000
3,NREM,forward,0.196024,0.172435,0.219997,1.000
4,NREM,forward_minus_reversed,0.007973,-0.024024,0.034865,0.680
5,NREM,reversed,0.188485,0.152690,0.216533,1.000


In [2]:
colors = {'Awake': '#cc5544', 'NREM': '#4477aa'}
fig, axes = get_fig(RN=1, CN=2, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
for state in ['Awake', 'NREM']:
    ax = next(ax_loader)
    part = scores[(scores.state == state) & (scores.direction == 'forward_minus_reversed')].dx_r2_flat
    ax.hist(part, bins=25, color=colors[state], alpha=.85)
    ax.axvline(0, color='0.2', lw=1)
    ax.axvline(part.median(), color='white', lw=2, ls='--')
    ax.set(xlabel=r'Forward minus reversed $\Delta x$ $R^2$', ylabel='Bootstrap count', title=state)
fig.savefig(OUT / 'time_arrow_bootstrap.png')
fig.savefig(OUT / 'time_arrow_bootstrap.svg')
fig.savefig(OUT / 'time_arrow_bootstrap.pdf')
plt.show()

Awake consistently favors reversed prediction: median forward-minus-reversed R² is -0.025 with a 95% episode-bootstrap interval [-0.055, -0.010]. NREM has a small positive median of 0.008, but its interval [-0.024, 0.035] spans zero. No state therefore provides positive, bootstrap-supported evidence that the fitted lag operator follows the physical time arrow.